# Anime

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import plotly.graph_objects as go
import ast
import plotly.express as px

In [3]:
# load dataset
df_profiles = pd.read_csv("../../data/cleaned/profiles_cleaned.csv")
df_person_anime_works = pd.read_csv("../../data/cleaned/person_anime_works_cleaned.csv")
df_person_details = pd.read_csv("../../data/cleaned/person_details_cleaned.csv")
df_person_alternate_names = pd.read_csv("../../data/cleaned/person_alternate_names_cleaned.csv")



In [4]:
df_person_details["birthday"] = pd.to_datetime(df_person_details["birthday"], errors="coerce")
df_person_details["birth_year"] = df_person_details["birthday"].dt.year.astype("Int64")

df_person_details["favorites"] = pd.to_numeric(df_person_details["favorites"], errors="coerce")


## In quale aree del mondo gli anime sono maggiormente seguiti?


In [5]:
#
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

#Load data
profiles = pd.read_csv("../../data/cleaned/profiles_cleaned.csv")

# extract country from location
profiles["country"] = (
    profiles["location"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA})
    .str.split(",")
    .str[-1]
    .str.strip()
)

#country names
country_fix = {
    "Usa": "United States",
    "US": "United States",
    "United States Of America": "United States",
    "UK": "United Kingdom",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Russian Federation": "Russia",
    "Russia": "Russia",
    "South Korea": "South Korea",
    "Korea, Republic of": "South Korea",
    "Korea, Rep.": "South Korea",

}
profiles["country"] = profiles["country"].replace(country_fix)

profiles["country"] = profiles["country"].where(
    profiles["country"].isna(),
    profiles["country"].str.replace(r"\s+", " ", regex=True).str.title()
)
#unique users per country
country_counts = (
    profiles.dropna(subset=["country"])
            .groupby("country")["username"]
            .nunique()
            .reset_index(name="n_users")
)

#which countries are not recognised by Plotly?
valid_countries = px.data.gapminder()["country"].unique()
not_recognised = (
    country_counts.loc[~country_counts["country"].isin(valid_countries), ["country", "n_users"]]
    .sort_values("n_users", ascending=False)
)
not_recognised.head(30)

# log scale
country_counts["log_users"] = np.log10(country_counts["n_users"] + 1)

# choropleth map
fig = px.choropleth(
    country_counts,
    locations="country",
    locationmode="country names",
    color="log_users",
    hover_name="country",
    hover_data={"n_users": True, "log_users": ":.2f"},
    title="Dove gli anime sono maggiormente seguiti? (utenti per paese, scala log10)",
)

fig.update_layout(
    coloraxis_colorbar=dict(title="log10(utenti)", tickformat=".1f")
)

fig.update_geos(showland=True, landcolor="lightgrey", showocean=True, oceancolor="#e6f2ff")

#Layout
fig.update_layout(template="plotly_white", width=1100, height=600)
fig.show()


La mappa evidenzia paesi in cui il dataset registra una maggiore presenza di utenti appassionati di anime. L'intensità del colore cresce all'aumentare del numero di utenti unici per paese; l'uso della trasformazione log(n_users + 1) comprime le differenze estreme di valore e rende visibili anche paesi con valori più bassi.

#### A questo punto possiamo vedere come l'interesse per gli anime si è sviluppato durante il corso del tempo, considerando i paesi dove esse son maggiormente diffuse.

## Come si è evoluta nel tempo la crescita della community / nuovi utenti per gli anime nelle aree dove sono più concentrate?

In [6]:
# converto joined + extract year
profiles["joined"] = pd.to_datetime(profiles["joined"], errors="coerce")
profiles["joined_year"] = profiles["joined"].dt.year

#filter year
profiles = profiles[(profiles["joined_year"] >= 1917) & (profiles["joined_year"] <= 2024)]

#Top 6 countries
top_countries = (
    profiles.dropna(subset=["country"])
            .groupby("country")["username"]
            .nunique()
            .sort_values(ascending=False)
            .head(6)
            .index
)
profiles["country_group"] = profiles["country"].where(profiles["country"].isin(top_countries), "Altri")

# new users to year
trend = (
    profiles.dropna(subset=["joined_year", "country_group"])
            .groupby(["joined_year", "country_group"])["username"]
            .nunique()
            .reset_index(name="n_users")
)
#Rimuovo "Altri" che prima avevo inserito
trend_no_others = trend[trend["country_group"] != "Altri"].copy()

#line chart
fig = px.line(
    trend_no_others,
    x="joined_year",
    y="n_users",
    color="country_group",
    markers=True,
    title="Crescita della community nel tempo: nuovi utenti registrati per anno (top paesi)"
)

fig.update_layout(
    template="plotly_white",
    width=1100, height=600,
    xaxis_title="Anno",
    yaxis_title="Nuovi utenti"
)


fig.update_layout(
    legend_title_text="Area"
)

fig.show()


Il grafico mostra l'evoluzione nel tempo dei nuovi utenti registrati ogni anno nei 6 paesi con più utenti nel dataset;

#### E l'evoluzione a livello globale? Vediamo come si modifica a livello temporale ...

## Crescita della community nel tempo: nuovi utenti registrati per anno:

In [7]:
#load data
df = profiles.copy()
#convert joined to datetime and extract  year
df["joined"] = pd.to_datetime(df["joined"], errors="coerce")
df["joined_year"] = df["joined"].dt.year

users_per_year = (
    df.dropna(subset=["joined_year"])
      .assign(joined_year=lambda x: x["joined_year"].astype(int))
      .groupby("joined_year", observed=True)["username"]
      .nunique()
      .reset_index(name="new_users")
      .sort_values("joined_year")
)

users_per_year = users_per_year[users_per_year["joined_year"].between(2005, 2024)].copy()

#line chart
fig = px.line(
    users_per_year,
    x="joined_year",
    y="new_users",
    markers=True,
    title="Crescita della community nel tempo: nuovi utenti registrati per anno",
    labels={"joined_year": "Anno", "new_users": "Nuovi utenti"},
    color_discrete_sequence=["#1f4e79"]
)
fig.update_layout(template="plotly_white", width=1100, height=600)
fig.show()

“Il grafico mostra l’andamento dei nuovi utenti registrati ogni anno: si osserva una crescita nel tempo, con picchi marcati negli anni più recenti.”